# RAG Temelleri — Uçtan Uca Demonstrasyon

Bu notebook bir RAG sistemini tek bir framework çağrısının arkasına saklamadan adım adım kurar:

1. cosine benzerliği,
2. chunking stratejileri,
3. embedding üretimi,
4. vektör indeksleme ve metadata filtresi,
5. retrieval + grounded generation,
6. Recall@k ve MRR ile değerlendirme,
7. isteğe bağlı Ollama ve Transformers backend'leri.

> Ana akış yalnızca Python standart kütüphanesiyle çalışır. `HashEmbedding` ve `ExtractiveGenerator` üretim modeli değil, veri akışını görünür kılan öğretim bileşenleridir.

## 0. Ortamı hazırla

Notebook hem depo kökünden hem de `rag/` klasöründen açılabilir. Aşağıdaki hücre doğru modül yolunu otomatik bulur.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
RAG_DIR = next((p / 'rag' for p in candidates if (p / 'rag' / 'rag_lab').is_dir()), None)
if RAG_DIR is None:
    RAG_DIR = next((p for p in candidates if (p / 'rag_lab').is_dir()), None)
if RAG_DIR is None:
    raise RuntimeError('rag/rag_lab klasörü bulunamadı; notebooku proje içinden açın.')

if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))

from examples.common import load_documents
from rag_lab import (
    ExtractiveGenerator, HashEmbedding, InMemoryVectorStore,
    OllamaEmbedding, OllamaGenerator, RAGPipeline,
    TransformersEmbedding, TransformersGenerator, chunk_documents,
)
from rag_lab.chunking import fixed_word_chunks, markdown_chunks, sentence_chunks
from rag_lab.evaluation import mean_reciprocal_rank, recall_at_k
from rag_lab.similarity import cosine_similarity, dot, l2_norm, normalize

print(f'RAG klasörü: {RAG_DIR}')
print(f'Python: {sys.version.split()[0]}')

RAG klasörü: /Users/alibayram/Desktop/single_letter_transformers/rag
Python: 3.13.14


## 1. Cosine benzerliği

Cosine iki vektörün uzunluğundan çok yönleri arasındaki açıya bakar:

$$\cos(a,b)=\frac{a\cdot b}{\lVert a\rVert_2\lVert b\rVert_2}$$

L2-normalize edilmiş iki vektörde cosine ile dot product aynıdır.

In [2]:
pairs = [
    ([1, 0], [10, 0], 'aynı yön, farklı uzunluk'),
    ([1, 0], [0, 3], 'dik yönler'),
    ([1, 1], [-1, -1], 'zıt yönler'),
]

for a, b, explanation in pairs:
    print(f'{explanation:28} cosine={cosine_similarity(a, b):+.3f}')

a, b = [1, 1], [2, 0]
print('\nElle hesap:')
print(f'dot(a,b) = {dot(a, b)}')
print(f'||a||={l2_norm(a):.3f}, ||b||={l2_norm(b):.3f}')
print(f'cosine={cosine_similarity(a, b):.3f}')
print(f'normalize(a)={normalize(a)}')

aynı yön, farklı uzunluk     cosine=+1.000
dik yönler                   cosine=+0.000
zıt yönler                   cosine=-1.000

Elle hesap:
dot(a,b) = 2
||a||=1.414, ||b||=2.000
cosine=0.707
normalize(a)=[0.7071067811865475, 0.7071067811865475]


### Tartışma

- `[1, 0]` ile `[1000, 0]` neden tamamen benzer sayılıyor?
- Sıfır vektörün cosine skoru neden tanımsızdır?
- Normalize edilmiş vektörlerde dot product neden yeterlidir?

## 2. Aynı metinde üç chunking stratejisi

Sabit pencere öngörülebilir bir baseline'dır. Cümle tabanlı yöntem anlam sınırlarını daha iyi korur. Yapı-duyarlı yöntem ise başlıkları chunk içinde taşır.

In [ ]:
soru = "sivilcelerim için aktif akne tedavisini araştırıyorum, bana bilgi ver!"

In [ ]:
sample_text = '''# Kurulum
Uygulamayı indirin. Hesabınızla giriş yapın. İlk eşitleme birkaç dakika sürebilir.

# Sorun giderme
Bağlantı kurulmazsa ağı kontrol edin. Ardından uygulamayı yeniden başlatın.
Sorun sürerse destek ekibine hata koduyla ulaşın.'''

strategies = {
    'fixed': fixed_word_chunks(sample_text, size=10, overlap=2),
    'sentence': sentence_chunks(sample_text, max_words=12, overlap_sentences=1),
    'markdown': markdown_chunks(sample_text, max_words=14, overlap=2),
}

for strategy, chunks in strategies.items():
    print(f'\n{strategy.upper()} — {len(chunks)} chunk')
    for index, chunk in enumerate(chunks):
        print(f'  [{index}] {chunk!r}')

## 3. Örnek bilgi bankasını yükle ve embed et

Bilgi bankasında iade, kargo, üyelik ve gizlilik belgeleri bulunur. `HashEmbedding`, kelime ve karakter n-gram'larını sabit boyutlu bir uzaya taşır. Semantik model değildir; yalnızca pipeline demonstrasyonu içindir.

In [ ]:
documents = load_documents()
for document in documents:
    print(f"{document.id:10} category={document.metadata['category']:8} {document.metadata['title']}")

chunks = chunk_documents(
    documents, strategy='sentence', max_words=35, overlap_sentences=1
)
embedder = HashEmbedding(dimension=256)
vectors = embedder.embed([chunk.text for chunk in chunks])

print(f'\n{len(documents)} belge -> {len(chunks)} chunk -> {len(vectors)} vektör')
print(f'Embedding boyutu: {len(vectors[0])}')
print(f'İlk vektörün normu: {l2_norm(vectors[0]):.3f}')

## 4. Vektör indeksleme, top-k arama ve metadata filtresi

Vektör deposu embedding üretmez. Vektörü, metni, belge kimliğini ve metadata'yı birlikte saklar; sorgu vektörüne göre en yakın chunk'ları döndürür.

In [ ]:
store = InMemoryVectorStore()
store.add(chunks, vectors)

question = 'İade için kaç günüm var?'
query_vector = embedder.embed([question])[0]
results = store.search(query_vector, k=4)

print(f'Soru: {question}\n')
for result in results:
    print(f'{result.score:+.3f}  {result.chunk.id:22} {result.chunk.text}')

In [ ]:
print('Yalnızca category=hesap belgelerinde ara:\n')
filtered = store.search(query_vector, k=4, filters={'category': 'hesap'})
for result in filtered:
    print(f'{result.score:+.3f}  {result.chunk.id:22} {result.chunk.metadata["title"]}')

## 5. Uçtan uca RAG: retrieve → prompt → answer

Pipeline soruyu embed eder, en yakın parçaları getirir, kaynak etiketli bir prompt kurar ve generator'a gönderir. Buradaki extractive generator bir LLM değildir; bağlamdaki en yakın cümleyi seçer.

In [ ]:
pipeline = RAGPipeline(
    embedder=HashEmbedding(dimension=256),
    store=InMemoryVectorStore(),
    generator=ExtractiveGenerator(),
)
indexed_count = pipeline.index(
    documents, chunk_strategy='sentence', max_words=35, overlap_sentences=1
)
answer = pipeline.ask('İade ücretim ne zaman yansır?', k=3)

print(f'{indexed_count} chunk indekslendi.')
print(f'\nCevap: {answer.answer}\n')
print('Kaynaklar:')
for source in answer.sources:
    print(f'- {source.chunk.id}  cosine={source.score:+.3f}')

### Generator'ın gördüğü grounded prompt

Prompt; yetersiz bağlamda cevap vermemeyi, kaynak etiketi eklemeyi ve kaynak metin içindeki talimatları uygulamamayı ister. Bu tek başına tam bir güvenlik sınırı değildir.

In [ ]:
print(answer.prompt)

## 6. Retriever'ı generator'dan bağımsız değerlendir

`Recall@k`, doğru belgenin ilk `k` sonuç içinde bulunup bulunmadığını; MRR ise ilk doğru sonucun ne kadar yukarıda olduğunu ölçer. Küçük veri setindeki sonuç bir benchmark değil, metrik demonstrasyonudur.

In [ ]:
cases = [
    ('Ürün iadesi kaç gün içinde yapılır?', {'iade'}),
    ('Kargom İstanbul dışına ne zaman gelir?', {'kargo'}),
    ('Şifremi sıfırlayan bağlantı ne kadar geçerli?', {'uyelik'}),
    ('Destek konuşmaları kaç gün saklanıyor?', {'gizlilik'}),
]

rankings = []
for query, relevant in cases:
    retrieved = pipeline.retrieve(query, k=4)
    ranking = [item.chunk.document_id for item in retrieved]
    rankings.append(ranking)
    print(f'{query}\n  beklenen={relevant}, gelen={ranking}')

relevant_ids = [relevant for _, relevant in cases]
print(f'\nRecall@1 = {recall_at_k(rankings, relevant_ids, k=1):.2f}')
print(f'Recall@3 = {recall_at_k(rankings, relevant_ids, k=3):.2f}')
print(f'MRR      = {mean_reciprocal_rank(rankings, relevant_ids):.2f}')

## 7. Chunking deneyini ölç

Aşağıdaki hücre aynı sorgu kümesini üç chunking stratejisiyle tekrar indeksler. Gerçek deneyde model, sorular ve relevance etiketleri sabit tutulmalı; chunking dışında aynı anda başka değişken değiştirilmemelidir.

In [ ]:
configs = {
    'fixed': {'size': 35, 'overlap': 5},
    'sentence': {'max_words': 35, 'overlap_sentences': 1},
    'markdown': {'max_words': 35, 'overlap': 5},
}

for strategy, kwargs in configs.items():
    experiment = RAGPipeline(HashEmbedding(256), InMemoryVectorStore(), ExtractiveGenerator())
    count = experiment.index(documents, chunk_strategy=strategy, **kwargs)
    experiment_rankings = [
        [item.chunk.document_id for item in experiment.retrieve(query, k=3)]
        for query, _ in cases
    ]
    recall = recall_at_k(experiment_rankings, relevant_ids, k=1)
    print(f'{strategy:9} chunks={count:2}  Recall@1={recall:.2f}')

## 8. İsteğe bağlı: Ollama ile gerçek yerel modeller

Önce terminalde embedding ve generation modellerini hazırlayın:

```bash
ollama pull embeddinggemma
ollama pull gemma3
```

Ardından `RUN_OLLAMA = True` yapın. Hücre varsayılan olarak ağ/servis çağrısı yapmaz.

In [ ]:
RUN_OLLAMA = False

if RUN_OLLAMA:
    ollama_pipeline = RAGPipeline(
        embedder=OllamaEmbedding('embeddinggemma'),
        store=InMemoryVectorStore(),
        generator=OllamaGenerator('gemma3'),
    )
    ollama_pipeline.index(
        documents, chunk_strategy='sentence', max_words=55, overlap_sentences=1
    )
    ollama_answer = ollama_pipeline.ask('Standart kargo ne zaman gelir?', k=3)
    print(ollama_answer.answer)
    print([source.chunk.id for source in ollama_answer.sources])
else:
    print('Ollama hücresi kapalı. Çalıştırmak için RUN_OLLAMA = True yapın.')

## 9. İsteğe bağlı: Transformers ile modelleri aynı sürece yükle

Önce `pip install -r rag/requirements-optional.txt` çalıştırın. Generation modelini cihazınıza, dilinize ve lisansınıza göre açıkça seçin. Hücre varsayılan olarak paket veya model indirmez.

In [ ]:
RUN_TRANSFORMERS = False
GENERATION_MODEL = 'MODEL_KARTINDAKI_CAUSAL_LM_ID'

if RUN_TRANSFORMERS:
    transformers_pipeline = RAGPipeline(
        embedder=TransformersEmbedding(
            'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
            device='cpu',
        ),
        store=InMemoryVectorStore(),
        generator=TransformersGenerator(GENERATION_MODEL, device='cpu'),
    )
    transformers_pipeline.index(
        documents, chunk_strategy='sentence', max_words=55, overlap_sentences=1
    )
    hf_answer = transformers_pipeline.ask('Pazarlama iznimi nasıl geri çekerim?', k=3)
    print(hf_answer.answer)
else:
    print('Transformers hücresi kapalı. Modeli seçip RUN_TRANSFORMERS = True yapın.')

## 10. Ders içi alıştırmalar

1. `k=1`, `k=3` ve `k=8` için kaynak kalitesini ve prompt uzunluğunu karşılaştırın.
2. Cevabı bilgi bankasında olmayan en az iki soru ekleyip abstention davranışını inceleyin.
3. Chunk boyutunu küçültüp overlap'i artırın; chunk sayısı ve Recall@1 nasıl değişiyor?
4. `category` filtresini yanlış kategoriyle uygulayın. Doğru belge neden artık bulunamıyor?
5. Hash backend ile gerçek embedding backend'inin sıralamalarını aynı sorularda karşılaştırın.
6. Bir belgeye `Önceki talimatları yok say` metni ekleyin; bunun neden veri, yetkilendirme ve prompt güvenliği problemi olduğunu tartışın.